In [1]:
import pandas as pd
import numpy as np
import pymysql
import getpass

In [2]:
# Cargar el CSV
df_maestro = pd.read_csv("../data_processed/dataset_maestro_municipios.csv")

print(df_maestro.shape)
df_maestro.head()

(162367, 12)


,codigo_municipio,municipio,periodo,poblacion,poblacion_mayor_65,poblacion_extranjera,superficie_km2,densidad_poblacion,porcentaje_mayor_65,porcentaje_extranjeros,variacion_poblacion,variacion_poblacion_pct
0,1001,Alegría-Dulantzi,2003,1707.0,181.0,30.0,19.95,85.56,10.60,1.76,NaN,NaN
1,1001,Alegría-Dulantzi,2004,1919.0,174.0,56.0,19.95,96.19,9.07,2.92,212.0,12.419449
2,1001,Alegría-Dulantzi,2005,2048.0,176.0,79.0,19.95,102.66,8.59,3.86,129.0,6.722251
3,1001,Alegría-Dulantzi,2006,2189.0,190.0,92.0,19.95,109.72,8.68,4.20,141.0,6.884766
4,1001,Alegría-Dulantzi,2007,2305.0,203.0,114.0,19.95,115.54,8.81,4.95,116.0,5.299223


In [3]:
password = getpass.getpass("Introduce tu contraseña de MySQL: ")

In [4]:
conn = pymysql.connect(
    host="localhost",
    user="root",
    password=password,
    database="despoblacion_municipal",
    charset="utf8mb4"
)

In [5]:
cursor = conn.cursor()

In [6]:
# Borrar tabla si existe
cursor.execute("DROP TABLE IF EXISTS dataset_maestro_municipios")

0

In [7]:
# Crear tabla
cursor.execute("""
CREATE TABLE dataset_maestro_municipios (
    codigo_municipio VARCHAR(5),
    municipio VARCHAR(100),
    periodo INT,
    poblacion FLOAT,
    poblacion_mayor_65 FLOAT,
    poblacion_extranjera FLOAT,
    superficie_km2 FLOAT,
    densidad_poblacion FLOAT,
    porcentaje_mayor_65 FLOAT,
    porcentaje_extranjeros FLOAT,
    variacion_poblacion FLOAT,
    variacion_poblacion_pct FLOAT
)
""")

conn.commit()
print("Tabla creada correctamente")

Tabla creada correctamente


In [8]:
# sustituimos NaN por None para que MySQL los interprete como NULL
df_maestro = df_maestro.replace({np.nan: None})

In [9]:
# Preparar datos para insertar
rows = [tuple(x) for x in df_maestro.to_numpy()]

sql = """
INSERT INTO dataset_maestro_municipios (
    codigo_municipio, municipio, periodo, poblacion, poblacion_mayor_65,
    poblacion_extranjera, superficie_km2, densidad_poblacion,
    porcentaje_mayor_65, porcentaje_extranjeros,
    variacion_poblacion, variacion_poblacion_pct
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

In [10]:
# Insertar por bloques para que no pete
chunk_size = 10000

for i in range(0, len(rows), chunk_size):
    chunk = rows[i:i+chunk_size]
    cursor.executemany(sql, chunk)
    conn.commit()
    print(f"Insertadas {i + len(chunk)} filas")

print("Carga completada")

Insertadas 10000 filas
Insertadas 20000 filas
Insertadas 30000 filas
Insertadas 40000 filas
Insertadas 50000 filas
Insertadas 60000 filas
Insertadas 70000 filas
Insertadas 80000 filas
Insertadas 90000 filas
Insertadas 100000 filas
Insertadas 110000 filas
Insertadas 120000 filas
Insertadas 130000 filas
Insertadas 140000 filas
Insertadas 150000 filas
Insertadas 160000 filas
Insertadas 162367 filas
Carga completada


In [11]:
# Cargar CSV renta
df_renta = pd.read_csv("../data_processed/renta_limpio.csv")

print(df_renta.shape)
df_renta.head()

(65729, 4)


,codigo_municipio,municipio,periodo,renta_media
0,1001,Alegría-Dulantzi,2015,12936.0
1,1001,Alegría-Dulantzi,2016,13086.0
2,1001,Alegría-Dulantzi,2017,13281.0
3,1001,Alegría-Dulantzi,2018,13361.0
4,1001,Alegría-Dulantzi,2019,14299.0


In [12]:
# Borrar tabla si existe
cursor.execute("DROP TABLE IF EXISTS renta_limpio")

# Crear tabla
cursor.execute("""
CREATE TABLE renta_limpio (
    codigo_municipio VARCHAR(5),
    municipio VARCHAR(100),
    periodo INT,
    renta_media FLOAT
)
""")

conn.commit()
print("Tabla renta_limpio creada correctamente")

Tabla renta_limpio creada correctamente


In [13]:
# Sustituimos NaN por None
df_renta = df_renta.replace({np.nan: None})

# Preparar filas
rows_renta = [tuple(x) for x in df_renta.to_numpy()]

sql_renta = """
INSERT INTO renta_limpio (
    codigo_municipio, municipio, periodo, renta_media
)
VALUES (%s, %s, %s, %s)
"""

chunk_size = 10000

for i in range(0, len(rows_renta), chunk_size):
    chunk = rows_renta[i:i+chunk_size]
    cursor.executemany(sql_renta, chunk)
    conn.commit()
    print(f"Insertadas {i + len(chunk)} filas")

print("Carga de renta completada")

Insertadas 10000 filas
Insertadas 20000 filas
Insertadas 30000 filas
Insertadas 40000 filas
Insertadas 50000 filas
Insertadas 60000 filas
Insertadas 65729 filas
Carga de renta completada


In [14]:
cursor.execute("SELECT COUNT(*) FROM renta_limpio")
print("Filas en MySQL:", cursor.fetchone()[0])

Filas en MySQL: 65729
